# Training a Phone Assistant with Reinforcement Learning: LFM3-1.7B + GRPO with TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_grpo_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

Imagine an assistant app that runs [LFM3-1.7B](https://huggingface.co/LiquidAI/LFM3-1.7B) on the phone and can use 108 tools: calendar, alarms, messages, email, maps, music, settings, smart home, payments, and more. For every request, the model has to pick the right tools among many lookalikes (a text message or a chat message? a timer or a sleep timer?), fill their arguments correctly, ask for missing details, and answer directly when no tool applies.

This notebook trains it with reinforcement learning, using Group Relative Policy Optimization (GRPO) from [TRL](https://github.com/huggingface/trl) and LoRA. There is no supervised warm start: the model learns from a reward that compares its tool calls with verified gold calls. Generation dominates the cost of RL, so we recommend a Colab A100 GPU.

## 🎯 What you'll find

- **The app**: a fixed catalog of 108 tool schemas, in every prompt
- **A verifiable reward**: the model's calls are parsed and compared argument by argument with the gold calls
- **GRPO + LoRA**: the model learns from groups of its own attempts
- **Evaluation**: exact-match accuracy on requests that use the training tools, and on 10 tools that never appear in training

# 📦 Installation & Setup

We pin every library to the versions this notebook was tested with.

In [ ]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

In [ ]:
import json
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B is released in bfloat16. GPUs without bfloat16 support load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Loading the model from Transformers 🤗

LFM3-1.7B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="left")

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)
print("✅ Model loaded successfully!")

# 📱 The app: 108 tools

The dataset comes with the tool catalog (`tools.json`, standard function schemas) and, for the reward, the way each argument is compared with the gold value (`scoring.json`).

In [ ]:
from collections import Counter
from datasets import load_dataset
from huggingface_hub import hf_hub_download

dataset_id = "LiquidAI/phone-assistant-tools"
tools = json.load(open(hf_hub_download(dataset_id, "tools.json", repo_type="dataset")))
kinds = json.load(open(hf_hub_download(dataset_id, "scoring.json", repo_type="dataset")))

print(f"🧰 {len(tools)} tools, for example:")
for tool in tools[:3]:
    print(json.dumps(tool["function"], ensure_ascii=False)[:300])

## The requests

Each example is a user request with the phone's date, time, and locale, and the gold calls. The requests are in 16 languages, and the gold dates and times were computed in code from the context. The splits are:

- `train`: 2,500 requests, which use 98 of the 108 tools
- `test`: requests that use the same 98 tools
- `test_unseen`: requests for the 10 held-out tools, which are in every prompt but never the right answer in training. Five of them are lookalikes of training tools (`draft_email` vs `send_email`, `set_sleep_timer` vs `set_timer`, ...).
- `showcase`: a few hand-checked examples from `test_unseen`

The categories cover what an assistant has to get right: one call, several calls, lookalike traps, no tool needed, a missing detail to ask about, and requests that are only partly possible.

In [ ]:
dataset = load_dataset(dataset_id)
print(dataset)
print(Counter(dataset["train"]["category"]))

for category in ["multi", "clarify"]:
    example = next(e for e in dataset["train"] if e["category"] == category)
    print(f"\n🗂️ {category} ({example['language']}, {example['context']['datetime']}): {example['request']}")
    print(f"🎯 Gold calls: {example['calls']}")

## Build the prompts

The system prompt holds the rules and the context, and the chat template adds the 108 tool schemas (about 8,000 tokens). LFM3 writes tool calls in XML, and the tokenizer knows how to parse them back into structured calls with `tokenizer.parse_response`.

We use LFM3's non-reasoning mode (`enable_thinking=False`). Since every prompt carries the same catalog, we render the prompts ourselves and give TRL plain text prompts.

In [ ]:
from datetime import datetime

SYSTEM_PROMPT = """You are a helpful assistant running on the user's phone. You can call the phone's tools.
Current date and time: {weekday}, {date} {time} ({timezone}). User locale: {locale}.

Rules:
- Call every tool needed to complete the request, with arguments taken from the request and the context above. Fill optional arguments only when the user gives them; never invent values.
- Resolve relative dates and times ("tomorrow", "on Friday", "in 20 minutes") against the current date and time. Times are local, in the formats the tool descriptions give.
- If the request needs a tool but a required detail is missing, ask a short clarifying question instead of calling a tool.
- Answer small talk and general-knowledge questions directly, without tools. Use translate_text, calculate, convert_units and convert_currency for translations, arithmetic and conversions, and search_web or get_news_headlines for current or local information.
- If no tool can do what is asked, say so briefly instead of calling a different tool. If only part of the request is possible, do that part and say what you could not do.
- Keep free-text arguments (messages, notes, search queries) in the user's language."""

def build_prompt(example):
    now = datetime.fromisoformat(example["context"]["datetime"])
    system = SYSTEM_PROMPT.format(weekday=now.strftime("%A"), date=now.strftime("%Y-%m-%d"), time=now.strftime("%H:%M"), **example["context"])
    messages = [{"role": "system", "content": system}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False, enable_thinking=False)

prompt = build_prompt(dataset["train"][0])
print(f"📏 Prompt length: {len(tokenizer(prompt, add_special_tokens=False).input_ids):,} tokens")
print(prompt[-1200:])

## Compare calls with the gold calls

A prediction is correct when it makes exactly the gold calls, in any order. Arguments are compared by kind:

- numbers, booleans, and enums exactly
- dates and times as values (`19:00` is `19:00:00`)
- codes such as currencies and phone numbers, without spaces or dashes
- names, without case or accents
- free text (a message body, a search query), with some tolerance for wording

Optional arguments left at their default value count as absent.

In [ ]:
import ast
import itertools
import math
import operator
import re
import unicodedata
from datetime import date, time
from difflib import SequenceMatcher

def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = "".join(" " if unicodedata.category(c)[0] in "PSZ" else c for c in unicodedata.normalize("NFC", s))
    return " ".join(s.split())

def contained(a, b):
    short, long_ = sorted((a, b), key=len)
    return bool(short) and short in long_ and len(short) / len(long_) >= 0.5

def name_equal(a, b):
    a, b = norm(a), norm(b)
    return a == b or contained(a, b)

def text_similar(a, b):
    a, b = norm(a), norm(b)
    return a == b or contained(a, b) or (bool(a) and bool(b) and SequenceMatcher(None, a, b).ratio() >= 0.6)

def parse_when(value, kind):
    value = str(value).strip()
    patterns = {"datetime": r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(:\d{2})?", "date": r"\d{4}-\d{2}-\d{2}", "time": r"\d{1,2}:\d{2}(:\d{2})?"}
    if not re.fullmatch(patterns[kind], value):
        return None
    return {"datetime": datetime, "date": date, "time": time}[kind].fromisoformat(value.zfill(5) if kind == "time" and len(value) == 4 else value)

OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
             ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calculate(expression):
    """Safely evaluate an arithmetic expression, or return None."""
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return OPERATORS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp):
            return OPERATORS[type(node.op)](walk(node.operand))
        raise ValueError
    try:
        return float(walk(ast.parse(str(expression).replace("^", "**").replace("×", "*").replace("÷", "/"), mode="eval").body))
    except Exception:
        return None

def same_list(gold, pred, equal):
    return len(gold) == len(pred) and any(all(equal(g, p) for g, p in zip(gold, perm)) for perm in itertools.permutations(pred))

def same_value(kind, gold, pred):
    try:
        if kind == "bool":
            return isinstance(pred, bool) and gold == pred
        if kind in ("int", "number"):
            return isinstance(pred, (int, float)) and not isinstance(pred, bool) and abs(float(gold) - float(pred)) < 1e-6
        if kind == "enum":
            return isinstance(pred, str) and gold == pred.strip().lower()
        if kind == "code":
            return isinstance(pred, str) and re.sub(r"[\s\-().]", "", str(gold).casefold()) == re.sub(r"[\s\-().]", "", pred.casefold())
        if kind in ("datetime", "date", "time"):
            return parse_when(pred, kind) is not None and parse_when(gold, kind) == parse_when(pred, kind)
        if kind == "expr":
            g, p = calculate(gold), calculate(pred)
            return g is not None and p is not None and math.isclose(g, p, rel_tol=1e-9, abs_tol=1e-9)
        if kind == "name":
            return isinstance(pred, str) and name_equal(gold, pred)
        if kind == "text":
            return isinstance(pred, str) and text_similar(gold, pred)
        if kind == "list_enum":
            return isinstance(pred, list) and sorted(gold) == sorted(str(x).lower() for x in pred)
        if kind == "list_name":
            return isinstance(pred, list) and same_list(gold, [str(x) for x in pred], name_equal)
        if kind == "list_text":
            return isinstance(pred, list) and same_list(gold, [str(x) for x in pred], text_similar)
    except (TypeError, ValueError, OverflowError, ZeroDivisionError):
        return False
    return False

schemas = {tool["function"]["name"]: tool["function"]["parameters"] for tool in tools}

def canonical(name, arguments):
    """Fill schema defaults, drop empty optional values, and treat end_date == date as a single day."""
    schema = schemas[name]
    required = set(schema.get("required", []))
    out = {k: v for k, v in (arguments or {}).items() if k in required or v not in (None, "", [])}
    for k, prop in schema["properties"].items():
        if k not in out and "default" in prop:
            out[k] = prop["default"]
    if "end_date" in out and out.get("end_date") == out.get("date"):
        del out["end_date"]
    return out

def same_call(gold, pred):
    if gold["name"] != pred.get("name"):
        return False
    g, p = canonical(gold["name"], gold.get("arguments")), canonical(gold["name"], pred.get("arguments"))
    return set(g) == set(p) and all(k in kinds[gold["name"]] and same_value(kinds[gold["name"]][k], g[k], p[k]) for k in g)

def calls_match(gold, pred):
    """True when the prediction makes exactly the gold calls, in any order."""
    if len(gold) != len(pred) or any(call.get("name") not in schemas for call in pred):
        return False
    return any(all(same_call(g, p) for g, p in zip(gold, perm)) for perm in itertools.permutations(pred))

def predicted_calls(completion_ids, prompt):
    """Parse a completion into a list of {"name", "arguments"} calls."""
    message = tokenizer.parse_response(completion_ids, prefix=prompt, tools=tools)
    return [call["function"] for call in message.get("tool_calls") or []]

# A quick check on a gold example
example = dataset["train"][0]
gold = json.loads(example["calls"])
print(calls_match(gold, gold), calls_match(gold, []))

# 📏 Evaluate the model before training

We generate one answer per request and count exact matches, per category. To keep the notebook fast, we evaluate on 200 requests of each test split.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, split, n=200, batch_size=8):
    model.eval()
    examples = dataset[split].shuffle(seed=42).select(range(n))
    correct = {}
    for start in tqdm(range(0, n, batch_size), desc=f"Evaluating {split}"):
        batch = examples.select(range(start, min(start + batch_size, n)))
        prompts = [build_prompt(e) for e in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        for output, prompt, example in zip(outputs[:, inputs["input_ids"].shape[1]:], prompts, batch):
            output = output[output != tokenizer.pad_token_id]
            correct.setdefault(example["category"], []).append(calls_match(json.loads(example["calls"]), predicted_calls(output, prompt)))
    results = {category: sum(c) / len(c) for category, c in correct.items()}
    results["all"] = sum(sum(c) for c in correct.values()) / n
    return results

def show(results, title):
    print(title)
    for split, r in results.items():
        print(f"   {split:<12} exact match {r['all']:.1%} | " + " | ".join(f"{k} {v:.0%}" for k, v in sorted(r.items()) if k != "all"))

baseline = {split: evaluate(model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")

# 🎯 Group Relative Policy Optimization (GRPO + LoRA)

## The reward

For each request, GRPO samples a group of answers and compares their rewards: answers that score above the group's average are reinforced, the others discouraged. Our reward is:

- **1.0** when the calls exactly match the gold calls (for requests where the right answer is to make no call, when the model makes no call)
- **0.2 × F1** of the tool names otherwise, a small partial credit for picking the right tools with wrong arguments, which helps early in training

TRL passes the dataset columns (here `calls`) and the completion token ids to the reward function.

In [ ]:
def tool_name_f1(gold, pred):
    gold_names, pred_names = Counter(c["name"] for c in gold), Counter(c["name"] for c in pred)
    overlap = sum((gold_names & pred_names).values())
    return 0.0 if overlap == 0 else 2 * overlap / (sum(gold_names.values()) + sum(pred_names.values()))

def tool_call_reward(prompts, completion_ids, calls, **kwargs):
    rewards = []
    for prompt, ids, gold in zip(prompts, completion_ids, calls):
        gold, pred = json.loads(gold), predicted_calls(ids, prompt)
        rewards.append(1.0 if calls_match(gold, pred) else 0.2 * tool_name_f1(gold, pred))
    return rewards

## Balance the training requests

Only 15% of the training requests need no tool call (`no_tool` and `clarify`). Since most rewards come from making the right calls, RL alone teaches the model to always call a tool, even when a detail is missing and it should ask. We counter this by showing the no-call requests twice.

In [ ]:
from datasets import concatenate_datasets

no_call = dataset["train"].filter(lambda e: e["category"] in ("no_tool", "clarify"))
train_dataset = concatenate_datasets([dataset["train"], no_call]).shuffle(seed=42)
train_dataset = train_dataset.map(lambda e: {"prompt": build_prompt(e)}, remove_columns=["request", "context"])
print(f"📚 Training requests: {len(train_dataset)}")
print(Counter(train_dataset["category"]))

## Wrap the model with PEFT

We adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # optional: LFM3's per-head attention gate
    ],
    task_type="CAUSAL_LM",
)

## Configure training

- `num_generations=8`: the size of each group. With 32 answers per step, each step trains on 4 requests.
- `generation_batch_size=8`: the answers of one request are generated together. Each prompt is about 8,000 tokens, so generating all 32 answers at once would need much more memory.
- `max_completion_length=256`: tool calls and short answers fit easily.
- `temperature=1.0`: enough variety within a group for the reward to tell answers apart.
- `beta=0.04`: a small KL penalty that keeps the model close to the original one. Without it, the behaviors the reward rarely asks for (like asking a clarifying question) fade as training goes on. With LoRA, TRL gets the reference model by disabling the adapter, so it doesn't load a second copy.

TRL generates with Transformers here. It can also generate with [vLLM](https://huggingface.co/docs/trl/vllm_integration) (`use_vllm=True`), which is much faster when your vLLM version supports the model.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./LFM3-1.7B-phone-assistant",
    max_steps=150,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,  # <- 32 answers per step
    num_generations=8,
    generation_batch_size=8,  # <- generate one group at a time, since each prompt is about 8,000 tokens
    max_completion_length=256,
    temperature=1.0,
    learning_rate=1e-5,
    beta=0.04,  # <- KL penalty towards the original model
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=5,
    logging_steps=5,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=tool_call_reward,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

## Launch training

In [ ]:
print("🚀 Starting GRPO training...")
trainer.train()
print("🎉 GRPO training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

Let's look at how the reward changed during training (1.0 means every answer in the batch was exactly right):

In [ ]:
for log in trainer.state.log_history:
    if "reward" in log:
        print(f"Step {log['step']:>3}: reward {log['reward']:.2f}, mean length {log['completions/mean_length']:.0f} tokens")

# 📊 Evaluate the trained model

In [ ]:
final = {split: evaluate(trainer.model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")
show(final, "📏 After training")

# 🔍 Original vs fine-tuned model

To see what changed, we give the same requests to the original model and to the trained model. With LoRA, `disable_adapter()` temporarily turns the adapter off, so we don't need to load the original model again. The `showcase` split holds hand-checked requests for tools that never appeared in training.

In [ ]:
def answer(model, example):
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**inputs, max_new_tokens=512, do_sample=False)[0, inputs["input_ids"].shape[1]:]
    message = tokenizer.parse_response(output, prefix=prompt, tools=tools)
    calls = [call["function"] for call in message.get("tool_calls") or []]
    return calls, (message.get("content") or "").strip()

def describe(calls, text):
    return json.dumps(calls, ensure_ascii=False) if calls else f"(no call) {text[:150]}"

trainer.model.eval()
for example in dataset["showcase"]:
    gold = json.loads(example["calls"])
    with trainer.model.disable_adapter():
        original = answer(trainer.model, example)
    trained = answer(trainer.model, example)
    print(f"\n💬 {example['request']} ({example['language']}, {example['category']})")
    print(f"   🎯 Gold:       {describe(gold, 'ask for the missing detail or answer directly')}")
    print(f"   {'✅' if calls_match(gold, original[0]) else '❌'} Original:   {describe(*original)}")
    print(f"   {'✅' if calls_match(gold, trained[0]) else '❌'} Fine-tuned: {describe(*trained)}")

And the summary over all the evaluation requests:

In [ ]:
print(f"{'exact match':<36}{'original':>10}{'fine-tuned':>12}")
for split, name in [("test", "training tools"), ("test_unseen", "held-out tools")]:
    print(f"{name:<36}{baseline[split]['all']:>10.1%}{final[split]['all']:>12.1%}")
    for category in sorted(k for k in baseline[split] if k != "all"):
        print(f"{'  ' + category:<36}{baseline[split][category]:>10.0%}{final[split][category]:>12.0%}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-1.7B-phone-assistant-lora")

output_dir = "./LFM3-1.7B-phone-assistant-merged"
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# 🚀 Next steps

- **Your own app**: export your tools as function schemas, collect real requests, and label the right calls. The reward only needs a way to compare calls, so you don't need full demonstrations.
- **Train longer**: the accuracy keeps improving after this run (200 steps reach about 80% on the training tools and 76% on the held-out ones). Increase `max_steps`.
- **Multi-turn agents**: `GRPOTrainer` can also run your tools during generation (`tools=[...]` or `environment_factory=...`) and reward the final outcome. See the [TRL documentation](https://huggingface.co/docs/trl/grpo_trainer).